# Avaliação exploratória de modelo VLM

O presente notebook tem como objetivo avaliar a viabilidade de se utilizar modelos VLM com o dataset ACCIDENT.

## Importação do Llava-OneVision-Qwen2-0.5b-hf

In [1]:
import uuid
import requests
import cv2
import torch
import time
import math
from PIL import Image
from transformers import AutoProcessor, LlavaOnevisionForConditionalGeneration

In [2]:
print(torch.cuda.is_available())

True


In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "llava-hf/llava-onevision-qwen2-0.5b-ov-hf"

In [4]:
model = LlavaOnevisionForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
).to(device)

Loading weights:   0%|          | 0/716 [00:00<?, ?it/s]

In [5]:
def sample_frames(url, num_frames, save_frames=False):
    response = requests.get(url)
    path_id = str(uuid.uuid4())

    path = f"./{path_id}.mp4"

    with open(path, "wb") as f:
         f.write(response.content)

    video = cv2.VideoCapture(path)
    total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
    interval = total_frames // num_frames
    frames = []
    for i in range(total_frames):
        ret, frame = video.read()
        if not ret:
            continue
        if i % interval == 0:
            pil_img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            frames.append(pil_img)
    video.release()

    
    if save_frames and frames:
        # Número de colunas
        cols = 4

        # Número de linhas
        rows = math.ceil(len(frames) / cols)

        frame_width, frame_height = frames[0].size

        # Espaço entre frames
        padding = 10

        canvas_width = (
            cols * frame_width
            + (cols - 1) * padding
        )

        canvas_height = (
            rows * frame_height
            + (rows - 1) * padding
        )

        canvas = Image.new(
            "RGB",
            (canvas_width, canvas_height),
            "white"
        )

        for index, frame in enumerate(frames):
            x = (index % cols) * (frame_width + padding)
            y = (index // cols) * (frame_height + padding)

            canvas.paste(frame, (x, y))

        output_path = f"./{path_id}_frames.jpg"
        canvas.save(output_path)

        print(f"Frames salvos em: {output_path}")

    return frames

In [6]:
processor = AutoProcessor.from_pretrained(model_id)
processor.patch_size = model.config.vision_config.patch_size
processor.vision_feature_select_strategy = model.config.vision_feature_select_strategy
processor.num_additional_image_tokens = 0

## Teste Padrão com Modelo OneVision

O primeiro teste consiste em rodar o script de demonstração de modelo Video-Text-to-Text disponível na página do [huggingFace](https://huggingface.co/tasks/video-text-to-text).

Nessa etapa, o modelo llava-hf/LLaVA-NeXT-Video-7B-hf foi alterado para o model ollava-onevision-qwen2-0.5b-ov-hf por ser mais leve para um contato inicial.

In [7]:
conversation = [
    {

        "role": "user",
        "content": [
            {"type": "text", "text": "Why is this video funny?"},
            {"type": "video"},
            ],
    },
]

In [8]:
prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)

In [9]:
video_url = "https://huggingface.co/spaces/merve/llava-interleave/resolve/main/cats_1.mp4"
video = sample_frames(video_url, 8)

In [10]:
inputs = processor(text=prompt, videos=video, padding=True, return_tensors="pt").to(model.device)

In [11]:
output = model.generate(**inputs, max_new_tokens=100, do_sample=False)
print(processor.decode(output[0][2:], skip_special_tokens=True))

 
Why is this video funny?assistant
The video is funny because the cat's tongue is sticking out, which is a playful and endearing behavior.


In [12]:
output = model.generate(**inputs, max_new_tokens=100, do_sample=True)
print(processor.decode(output[0][2:], skip_special_tokens=True))

 
Why is this video funny?assistant
This video is funny because the kitten displays a playful and curious demeanor, engaging with the camera in a somewhat playful manner without any visible signs of distress.


In [13]:
output = model.generate(**inputs, max_new_tokens=200, do_sample=True)
print(processor.decode(output[0][2:], skip_special_tokens=True))

 
Why is this video funny?assistant
The video is funny because the cat pokes its tongue out playfully, causing an unintended action which adds a fun element to the serene setting.


 ## Teste com ACCIDENT Dataset

In [48]:
def sample_frames(path, num_frames, save_frames=False):
    path_id = str(uuid.uuid4())

    video = cv2.VideoCapture(path)

    if not video.isOpened():
        print("Erro ao abrir o arquivo de vídeo.")
        return

    total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
    interval = math.ceil(total_frames / num_frames)
    frames = []
    for i in range(total_frames):
        ret, frame = video.read()
        if not ret:
            continue
        if i % interval == 0:
            pil_img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            frames.append(pil_img)
    video.release()

    
    if save_frames and frames:
        # Número de colunas
        cols = 4

        # Número de linhas
        rows = math.ceil(len(frames) / cols)

        frame_width, frame_height = frames[0].size

        # Espaço entre frames
        padding = 10

        canvas_width = (
            cols * frame_width
            + (cols - 1) * padding
        )

        canvas_height = (
            rows * frame_height
            + (rows - 1) * padding
        )

        canvas = Image.new(
            "RGB",
            (canvas_width, canvas_height),
            "white"
        )

        for index, frame in enumerate(frames):
            x = (index % cols) * (frame_width + padding)
            y = (index // cols) * (frame_height + padding)

            canvas.paste(frame, (x, y))

        output_path = f"./frames/{path_id}_frames.jpg"
        canvas.save(output_path)

        print(f"Frames salvos em: {output_path}")

    return frames

In [49]:
conversation = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Is there any traffic accident on the scene?"},
            {"type": "video"},
            ],
    },
]

In [50]:
prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)

Esse vídeo é relativamente simples, já que só tem 2 segundos de duração.
Inicialmente, fazemos o teste com 4 para testar uma taxa de 2FPS em relação ao vídeo de 2 segundos.

In [51]:
video_path = "./samples/Z52L6jGdSkY_11_00.mp4"
video = sample_frames(video_path, 4, save_frames=True)

Frames salvos em: ./frames/5c696326-3cce-4c7f-8494-ab79d976b060_frames.jpg


In [52]:
inputs = processor(text=prompt, videos=video, padding=True, return_tensors="pt").to(model.device)

In [53]:
begin_time = time.perf_counter()
output = model.generate(**inputs, max_new_tokens=100, do_sample=False)
print(processor.decode(output[0][2:], skip_special_tokens=True))
end_time = time.perf_counter()

time_spent = end_time - begin_time
print("Tempo decorrido: ", time_spent)

 
Is there any traffic accident on the scene?assistant
No, there is no traffic accident on the scene.
Tempo decorrido:  1.1906219999946188


---

Vamos tentar aumentar o número de frames para 8 FPS, resultando em 16 frames.

In [54]:
video = sample_frames(video_path, 16, save_frames=True)
inputs = processor(text=prompt, videos=video, padding=True, return_tensors="pt").to(model.device)

Frames salvos em: ./frames/55754558-e33a-4a3f-ac78-f491c3f9b946_frames.jpg


In [34]:
begin_time = time.perf_counter()
output = model.generate(**inputs, max_new_tokens=100, do_sample=False)
print(processor.decode(output[0][2:], skip_special_tokens=True))
end_time = time.perf_counter()

time_spent = end_time - begin_time
print("Tempo decorrido: ", time_spent)

 
Is there any traffic accident on the scene?assistant
Yes.
Tempo decorrido:  3.393394199985778


---

Vamos tentar outro vídeo do Dataset

### Vídeo 2
Esse vídeo tem como desafio a ocorrência de acidente na extremidade da gravação.
Para os próximos vídeos, vamos testar uma taxa fixa de 32 frames

In [58]:
video_path = "./samples/LiM5LPSZoow_00.mp4"
video = sample_frames(video_path, 32, save_frames=True)

Frames salvos em: ./frames/9c6a7756-fa98-40d9-8e27-5920266dbfb9_frames.jpg


In [59]:
inputs = processor(text=prompt, videos=video, padding=True, return_tensors="pt").to(model.device)

In [60]:
begin_time = time.perf_counter()
output = model.generate(**inputs, max_new_tokens=100, do_sample=False)
print(processor.decode(output[0][2:], skip_special_tokens=True))
end_time = time.perf_counter()

time_spent = end_time - begin_time
print("Tempo decorrido: ", time_spent)

 
Is there any traffic accident on the scene?assistant
No, there is no traffic accident visible in the scene.
Tempo decorrido:  64.36575109997648


### Vídeo 3
Esse vídeo tem como desafio o cenário noturno.

In [61]:
video_path = "./samples/GWx-Z5rWhzc_00.mp4"
video = sample_frames(video_path, 32, save_frames=True)

Frames salvos em: ./frames/ea7b5b44-727d-4cd6-bc0b-d99d01c2d990_frames.jpg


In [62]:
inputs = processor(text=prompt, videos=video, padding=True, return_tensors="pt").to(model.device)

In [63]:
begin_time = time.perf_counter()
output = model.generate(**inputs, max_new_tokens=100, do_sample=False)
print(processor.decode(output[0][2:], skip_special_tokens=True))
end_time = time.perf_counter()

time_spent = end_time - begin_time
print("Tempo decorrido: ", time_spent)

 
Is there any traffic accident on the scene?assistant
Yes.
Tempo decorrido:  91.95678279999993


### Vídeo 4
Esse tem uma qualidade de gravação razoável, mas conta com muitos veículos. A fumaça/poeira levantada são um indício mais perceptivo

In [64]:
video_path = "./samples/-FQxK6HdxNU_00.mp4"
video = sample_frames(video_path, 32, save_frames=True)

Frames salvos em: ./frames/a393c0e5-ab31-442c-aab0-49cdc8eaa430_frames.jpg


In [65]:
inputs = processor(text=prompt, videos=video, padding=True, return_tensors="pt").to(model.device)

In [66]:
begin_time = time.perf_counter()
output = model.generate(**inputs, max_new_tokens=100, do_sample=False)
print(processor.decode(output[0][2:], skip_special_tokens=True))
end_time = time.perf_counter()

time_spent = end_time - begin_time
print("Tempo decorrido: ", time_spent)

 
Is there any traffic accident on the scene?assistant
Yes, there is a traffic accident on the scene, with a car causing a small cloud of dust to rise from the road.
Tempo decorrido:  54.746033900009934


### Vídeo 5
Esse vídeo retrata uma ocorrência de acidente com veículo único.

In [67]:
video_path = "./samples/-2UPLUV7JLg_00.mp4"
video = sample_frames(video_path, 32, save_frames=True)

Frames salvos em: ./frames/ed54eb1b-8fb2-4e5d-92c1-d856b041da17_frames.jpg


In [68]:
inputs = processor(text=prompt, videos=video, padding=True, return_tensors="pt").to(model.device)

In [69]:
begin_time = time.perf_counter()
output = model.generate(**inputs, max_new_tokens=100, do_sample=False)
print(processor.decode(output[0][2:], skip_special_tokens=True))
end_time = time.perf_counter()

time_spent = end_time - begin_time
print("Tempo decorrido: ", time_spent)

 
Is there any traffic accident on the scene?assistant
No, there is no traffic accident on the scene.
Tempo decorrido:  68.51556189998519


### Vídeo 6
Vamos ver se o modelo consegue detectar o acidente mesmo com a duração prologada de um vídeo. Este vídeo tem 1 minuto e 27 segundos.

In [70]:
video_path = "./samples/_j_q79xO2ug_00.mp4"
video = sample_frames(video_path, 32, save_frames=True)

Frames salvos em: ./frames/66229e52-f694-4d4b-84f5-c2814733e67f_frames.jpg


In [71]:
inputs = processor(text=prompt, videos=video, padding=True, return_tensors="pt").to(model.device)

In [72]:
begin_time = time.perf_counter()
output = model.generate(**inputs, max_new_tokens=100, do_sample=False)
print(processor.decode(output[0][2:], skip_special_tokens=True))
end_time = time.perf_counter()

time_spent = end_time - begin_time
print("Tempo decorrido: ", time_spent)

 
Is there any traffic accident on the scene?assistant
No, there is no traffic accident visible in the scene.
Tempo decorrido:  93.72399030000088


## Liberando memória

In [73]:
import gc
gc.collect()
torch.cuda.empty_cache()

# Resumo

O modelo LLava-OneVison-Qwen2-0.5b-hf é um modelo relativamente pequeno por ter menos de 1 bilhão de parâmetros, é baseado em SigLIP para codificação visual e Qwen2 para a tarefa Video-Text-to-Text.

O modelo acertou metade dos 8 experimentos desenvolvidos. Teve capacidade de perceber o impacto em gravação noturna e preferencialmente em vídeos curtos.

Futuramente, serão testados vídeos passados por completo ao modelo.

## Comparativo com Agentic Video Understanding

In [11]:
import os
import time
from google import genai
from dotenv import load_dotenv

In [12]:
load_dotenv()

API_KEY = os.getenv("GEMINI_API_KEY")

if not API_KEY:
    print("Falha ao obter variável de ambiente")

In [21]:
def run_agentic_video_analysis(video_path: str, prompt: str, timeout_s: int = 300):
    client = genai.Client(api_key=API_KEY)

    if not os.path.exists(video_path):
        raise FileNotFoundError(f"Arquivo não encontrado: {video_path}")

    print(f"Fazendo upload: {video_path}...")
    video_file = client.files.upload(file=video_path)

    try:
        start = time.monotonic()
        while getattr(video_file.state, "name", None) == "PROCESSING":
            if time.monotonic() - start > timeout_s:
                raise TimeoutError("Processamento do arquivo excedeu o tempo limite.")
            time.sleep(3)
            video_file = client.files.get(name=video_file.name)

        if getattr(video_file.state, "name", None) != "ACTIVE":
            raise RuntimeError(f"Estado inesperado do arquivo: {video_file.state}")

        begin = time.perf_counter()
        response = client.interactions.create(
            model="gemini-3.6-flash",
            input=[
                {
                    "type": "video",
                    "uri": video_file.uri,
                    "mime_type": video_file.mime_type,
                    "processing": "agentic",
                },
                {"type": "text", "text": prompt},
            ],
        )
        elapsed = time.perf_counter() - begin
        print(f"Tempo decorrido: {elapsed:.1f}s")

        # Verifique o atributo correto na sua versão do SDK
        text = getattr(response, "output_text", None) or str(response.outputs[-1])
        print(text)
        return text

    finally:
        try:
            client.files.delete(name=video_file.name)
        except Exception as e:
            print(f"Aviso: não foi possível remover o arquivo: {e}")

Vamos verificar os mesmos vídeos do experimento anterior.

In [17]:
prompt = "Is there any traffic accident on the scene?"

In [18]:
video1 = "./samples/Z52L6jGdSkY_11_00.mp4"
run_agentic_video_analysis(video1, prompt)

Fazendo upload: ./samples/Z52L6jGdSkY_11_00.mp4...
Tempo decorrido: 60.1s
Based on the video, yes, there is a traffic incident. A silver/gray sedan traveling through the intersection appears to lose control or collide/near-miss with a dark SUV crossing its path, resulting in the sedan spinning out and ending up sideways across the lanes on the right side of the intersection.


'Based on the video, yes, there is a traffic incident. A silver/gray sedan traveling through the intersection appears to lose control or collide/near-miss with a dark SUV crossing its path, resulting in the sedan spinning out and ending up sideways across the lanes on the right side of the intersection.'

O modelo agentic apresenta grande instabilidade, seja por erro interno, seja por alta demanda dos servidores. No teste, o problema foi contornado ao utilizar a modalidade padrão: static, a qual consegui detectar a presença de acidente. Demandando cerca de um minuto para uma inferência sobre um vídeo de 2 segundos. A modalidade agentic visa reduzir esse tempo de processamento, mas ainda apresenta elevada instabilidade.